# Clustering — Niveau 1 (intra-foyer) — 20 foyers REFIT + réplication UCI

## Objectif

Répliquer, **foyer par foyer**, le protocole de clustering complet validé dans le TFC
(`04_UCI_clustering.pdf` : Cycle 1 — K-Means / K-Medoids / HAC-Ward, puis
Cycle 2 — GMM / DBSCAN, sélection multicritères), afin de répondre à la question centrale
de l'article :

> **La structure à 3 régimes (semaine standard / week-end / faible consommation) trouvée sur
> le foyer français UCI est-elle un artefact d'un seul foyer, ou un phénomène robuste et
> généralisable à d'autres foyers, dans un autre pays ?**

Le foyer UCI n'est pas recalculé ici (il l'a déjà été dans le TFC) — son résultat connu est
intégré comme **réplication de référence** dans le tableau de synthèse final.

**Note de rigueur méthodologique :** le TFC original sélectionnait le meilleur modèle final en
comparant des « scores globaux » renormalisés séparément dans chaque famille d'algorithme
(non comparables entre familles). Ce notebook corrige ce point : la sélection finale entre
familles d'algorithmes se fait sur le **Silhouette Score brut** (directement comparable),
voir `src/clustering_pipeline.py::select_best_across_algorithms`.

AUTEUR : ENANGA DANIEL CHRISTIAN


## Étape 0 : Imports

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
matplotlib.rcParams['figure.dpi'] = 120

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(BASE_DIR, "src"))

from features import FEATURES_FINAL
from clustering_pipeline import (
    run_cycle1, run_cycle2, select_best_across_algorithms,
    refit_final_model, reorder_labels_by_size, evaluate_clustering,
)

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "refit")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
IMAGES_DIR = os.path.join(BASE_DIR, "images", "clustering_level1")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

print("Bibliothèques et pipeline de clustering chargés.")


## Étape 1 : Chargement des features + résultat UCI de référence

Le résultat UCI ci-dessous est repris **tel quel** du TFC original (`04_UCI_clustering.pdf`,
modèle final GMM covariance `full`, k=3) — il n'est pas recalculé, seulement intégré comme
ligne de référence dans le tableau de synthèse.

In [ ]:
level1_path = os.path.join(PROCESSED_DIR, "level1_features_raw.csv")
df_level1_raw = pd.read_csv(level1_path, parse_dates=["date"]) if os.path.exists(level1_path) else pd.DataFrame()

if df_level1_raw.empty:
    print("[ATTENTION] level1_features_raw.csv introuvable ou vide -- exécutez d'abord le Notebook 02.")

# --- Résultat UCI de référence (TFC original, section 5.3, Tab. 21) ---
UCI_REFERENCE_ROW = {
    "house_id": "UCI_FR",
    "n_days": 1431,
    "best_algorithm": "GMM",
    "k": 3,
    "covariance_type": "full",
    "silhouette": 0.2422,
    "davies_bouldin": 1.4836,
    "calinski_harabasz": 192.43,
    "is_source": "TFC original (référence, non recalculé)",
}
print("Réplication de référence (UCI, France) :")
print(UCI_REFERENCE_ROW)


## Étape 2 : Fonction de traitement d'un foyer (Cycle 1 + Cycle 2 + sélection)

Cette fonction encapsule exactement le protocole du TFC (`k` exploré de 2 à 10, K-Means avec
10 initialisations, GMM avec 4 types de covariance, DBSCAN par grille), afin d'être appelée
identiquement sur chacun des 20 foyers.

In [ ]:
MIN_DAYS_FOR_CLUSTERING = 30  # en dessous, le clustering intra-foyer est peu fiable (avertissement, pas exclusion)


def process_house(df_house: pd.DataFrame, house_id: str, k_range=range(2, 11), random_state: int = 42) -> dict:
    """
    Exécute le protocole complet de clustering (Cycle 1 + Cycle 2 + sélection
    finale) sur les signatures journalières d'UN foyer.

    Retourne un dict contenant la ligne de synthèse et le DataFrame étiqueté
    (une ligne par jour, avec son cluster final).
    """
    X_raw = df_house[FEATURES_FINAL].values
    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)

    cycle1 = run_cycle1(X, k_range=k_range, random_state=random_state)
    cycle2 = run_cycle2(X, k_range=k_range, random_state=random_state)
    best = select_best_across_algorithms(cycle1, cycle2)

    _, labels = refit_final_model(X, best["best_algorithm"], best, random_state=random_state)
    labels_ordered = reorder_labels_by_size(labels)

    synthesis_row = {
        "house_id": house_id,
        "n_days": df_house.shape[0],
        "best_algorithm": best["best_algorithm"],
        "k": int(best.get("k", np.nan)) if not pd.isna(best.get("k", np.nan)) else np.nan,
        "covariance_type": best.get("covariance_type", None),
        "silhouette": best["silhouette"],
        "davies_bouldin": best["davies_bouldin"],
        "calinski_harabasz": best["calinski_harabasz"],
        "is_source": "REFIT (calculé)",
    }

    df_labeled = df_house[["house_id", "date"]].copy()
    df_labeled["cluster"] = labels_ordered

    return {"synthesis": synthesis_row, "labels": df_labeled, "cycle1": cycle1, "cycle2": cycle2}


print("Fonction process_house() définie.")


## Étape 3 : Exécution sur les 20 foyers REFIT

In [ ]:
synthesis_rows = []
labels_frames = []
house_pipeline_results = {}  # conservé pour inspection ultérieure (ex: notebook explicabilité)

if not df_level1_raw.empty:
    house_ids = sorted(df_level1_raw["house_id"].unique())
    print(f"Traitement de {len(house_ids)} foyers REFIT...\n")

    for house_id in house_ids:
        df_house = df_level1_raw[df_level1_raw["house_id"] == house_id].reset_index(drop=True)

        if df_house.shape[0] < MIN_DAYS_FOR_CLUSTERING:
            print(f"[ATTENTION] {house_id} : seulement {df_house.shape[0]} jours -- résultat à interpréter avec prudence.")

        print(f"--- {house_id} ({df_house.shape[0]} jours) ---")
        result = process_house(df_house, house_id)
        synthesis_rows.append(result["synthesis"])
        labels_frames.append(result["labels"])
        house_pipeline_results[house_id] = result
        s = result["synthesis"]
        print(f"    Meilleur algorithme : {s['best_algorithm']} (k={s['k']}) | Silhouette={s['silhouette']:.4f}\n")

    print("Traitement terminé pour tous les foyers REFIT.")
else:
    print("[ATTENTION] Aucune donnée à traiter.")


## Étape 4 : Tableau de synthèse (21 lignes = 20 foyers REFIT + réplication UCI)

**C'est le tableau le plus important de l'article** — il répond directement à la question de
généralisation posée en introduction de ce notebook.

In [ ]:
if synthesis_rows:
    df_synthesis = pd.DataFrame(synthesis_rows + [UCI_REFERENCE_ROW])
    df_synthesis = df_synthesis.sort_values("silhouette", ascending=False).reset_index(drop=True)

    synthesis_out_path = os.path.join(RESULTS_DIR, "level1_synthesis_table.csv")
    df_synthesis.to_csv(synthesis_out_path, index=False)

    print(f"Tableau de synthèse sauvegardé : {synthesis_out_path}\n")
    print(df_synthesis.to_string(index=False))

    print("\n=== Répartition des algorithmes retenus comme meilleurs (hors UCI) ===")
    print(df_synthesis[df_synthesis['house_id'] != 'UCI_FR']['best_algorithm'].value_counts())


## Étape 5 : Visualisation comparative — Silhouette par foyer

In [ ]:
if synthesis_rows:
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ["#C73E1D" if h == "UCI_FR" else "#2E86AB" for h in df_synthesis["house_id"]]
    ax.bar(df_synthesis["house_id"], df_synthesis["silhouette"], color=colors)
    ax.set_title("Silhouette Score du meilleur modèle — par foyer (Niveau 1)", fontweight="bold")
    ax.set_ylabel("Silhouette Score")
    ax.set_xlabel("Foyer (rouge = référence UCI, France)")
    plt.xticks(rotation=60, ha="right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, "01_silhouette_par_foyer.png"), dpi=300, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée : 01_silhouette_par_foyer.png")


## Étape 6 : Sauvegarde des labels journaliers par foyer

In [ ]:
if labels_frames:
    df_all_labels = pd.concat(labels_frames, ignore_index=True)
    labels_out_path = os.path.join(RESULTS_DIR, "level1_cluster_labels.csv")
    df_all_labels.to_csv(labels_out_path, index=False)
    print(f"Labels journaliers sauvegardés : {labels_out_path} ({df_all_labels.shape[0]} lignes)")


## Étape 7 : Profils de charge horaires par cluster (ce qui manquait pour "voir" les régimes)

Le tableau de synthèse dit *combien* de régimes existent et à quel point ils sont séparés
(Silhouette). Cette section montre *à quoi ils ressemblent* : la courbe de charge moyenne sur
24h de chaque cluster, avec son enveloppe de variabilité (±1 écart-type) — exactement l'esprit
de la Figure 19 du TFC original (foyer UCI), généralisée ici à plusieurs foyers REFIT.

Cellule volontairement autonome (relit les fichiers sur disque) : tu peux l'exécuter seule,
sans relancer toute la boucle de clustering ci-dessus.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(BASE_DIR, "src"))
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "refit")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
IMAGES_DIR = os.path.join(BASE_DIR, "images", "clustering_level1")

df_synthesis = pd.read_csv(os.path.join(RESULTS_DIR, "level1_synthesis_table.csv"))
df_all_labels = pd.read_csv(os.path.join(RESULTS_DIR, "level1_cluster_labels.csv"), parse_dates=["date"])
df_all_labels["date"] = df_all_labels["date"].dt.date


def plot_hourly_profiles_by_cluster(power_kw_minute: pd.Series, df_labels_house: pd.DataFrame, title: str, ax):
    """Trace, pour UN foyer, la courbe de charge moyenne sur 24h de chaque
    cluster (moyenne ± 1 écart-type), à partir de sa série minute et de ses
    labels journaliers."""
    df = power_kw_minute.to_frame("power_kw")
    df["date"] = df.index.date
    df["hour"] = df.index.hour
    df = df.merge(df_labels_house[["date", "cluster"]], on="date", how="inner")

    for cluster in sorted(df["cluster"].unique()):
        sub = df[df["cluster"] == cluster]
        hourly = sub.groupby("hour")["power_kw"]
        mean_profile, std_profile = hourly.mean(), hourly.std()
        n_days = sub["date"].nunique()
        ax.plot(mean_profile.index, mean_profile.values, linewidth=2, label=f"Cluster {cluster} (n={n_days} j)")
        ax.fill_between(mean_profile.index, (mean_profile - std_profile).clip(lower=0),
                         mean_profile + std_profile, alpha=0.15)

    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel("Heure de la journée")
    ax.set_ylabel("Puissance (kW)")
    ax.set_xticks(range(0, 24, 4))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)


# Trois foyers illustratifs et contrastés : la meilleure séparabilité, la plus faible,
# et un foyer à structure plus riche (k > 2) -- adapte cette liste à ta guise.
illustrative_houses = (
    df_synthesis[df_synthesis["house_id"] != "UCI_FR"]
    .sort_values("silhouette", ascending=False)
)
houses_to_plot = []
if not illustrative_houses.empty:
    houses_to_plot.append(illustrative_houses.iloc[0]["house_id"])   # meilleure Silhouette
    houses_to_plot.append(illustrative_houses.iloc[-1]["house_id"])  # plus faible Silhouette
    higher_k = illustrative_houses[illustrative_houses["k"] > 2]
    if not higher_k.empty:
        houses_to_plot.append(higher_k.iloc[0]["house_id"])          # un foyer k>2, structure plus riche

fig, axes = plt.subplots(1, len(houses_to_plot), figsize=(6 * len(houses_to_plot), 5))
if len(houses_to_plot) == 1:
    axes = [axes]

for ax, house_id in zip(axes, houses_to_plot):
    num = house_id.replace("REFIT_", "")
    series_path = os.path.join(PROCESSED_DIR, f"house_{num}_clean.csv")
    if not os.path.exists(series_path):
        ax.set_visible(False)
        continue
    series = pd.read_csv(series_path, index_col=0, parse_dates=True)["power_kw"]
    df_labels_house = df_all_labels[df_all_labels["house_id"] == house_id]
    k = df_synthesis.loc[df_synthesis["house_id"] == house_id, "k"].values[0]
    sil = df_synthesis.loc[df_synthesis["house_id"] == house_id, "silhouette"].values[0]
    plot_hourly_profiles_by_cluster(series, df_labels_house, f"{house_id} (k={k}, Silhouette={sil:.3f})", ax)

plt.suptitle("Profils de charge horaires moyens par cluster — Niveau 1 (foyers illustratifs)", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "02_profils_horaires_par_cluster.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : 02_profils_horaires_par_cluster.png")
print("\nPour voir TOUS les foyers plutôt que 3 : remplace houses_to_plot par")
print("df_synthesis['house_id'] (hors 'UCI_FR') et affiche par lots de 4-5 dans une grille.")


## Résumé final

**Fichiers produits :**
- `results/level1_synthesis_table.csv` — tableau comparatif à 21 lignes (résultat principal)
- `results/level1_cluster_labels.csv` — cluster journalier par foyer
- `images/clustering_level1/01_silhouette_par_foyer.png`
- `images/clustering_level1/02_profils_horaires_par_cluster.png` — à quoi ressemblent les régimes

**Étape suivante :** Notebook `04_Clustering_Level2_Household_Typology.ipynb` — construction
des signatures ménages agrégées et **véritable segmentation des ménages** (n=20), validée par
les métadonnées socio-démographiques REFIT.
